# Notebook 2: Mask2Former Inference with DINOv3 ViT-7B

**Paper Reference**: DINOv3, Section 6.3.2 — Semantic Segmentation with Mask2Former

We run inference using the **pretrained** Mask2Former decoder on top of a **frozen DINOv3 ViT-7B** backbone.
No training here — just loading weights and evaluating on ADE20k.

**Paper Results (Table 11)**:

| Setting | mIoU |
|---------|------|
| Single-scale | 62.6 |
| Multi-scale TTA | 63.0 |

**Hardware Requirement**: H100 80GB (the 7B model + decoder needs ~45GB VRAM at resolution 896)

## 1. Setup

In [ ]:
import torch, os, sys

# Verify GPU has enough VRAM
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')

if vram_gb < 60:
    print('\nWARNING: The 7B M2F segmentor requires ~45-50GB VRAM.')
    print('You need at least an A100 80GB or H100 80GB.')
else:
    print('\nVRAM is sufficient for the 7B model.')

In [ ]:
# Clone and install DINOv3
!git clone https://github.com/facebookresearch/dinov3.git 2>/dev/null || echo 'Already cloned'
%cd dinov3
!pip install -e . -q
%cd ..

sys.path.insert(0, 'dinov3')
sys.path.insert(0, 'dinov3-segmentation-study')

In [ ]:
# Download ADE20k (if not already present from Notebook 1)
if not os.path.exists('data/ADEChallengeData2016'):
    !mkdir -p data
    !wget -q http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip -O data/ade20k.zip
    !unzip -q data/ade20k.zip -d data/ && rm data/ade20k.zip
    print('ADE20k downloaded.')
else:
    print('ADE20k already present.')

In [ ]:
# Download ViT-7B backbone + Mask2Former decoder weights
# Request access at: https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/

VIT7B_URL = '<PASTE_7B_BACKBONE_URL>'     # Replace with your URL
M2F_URL   = '<PASTE_M2F_HEAD_URL>'         # Replace with your URL

!mkdir -p weights
if not os.path.exists('weights/dinov3_vit7b16.pth'):
    !wget -q -O weights/dinov3_vit7b16.pth "{VIT7B_URL}"
if not os.path.exists('weights/dinov3_vit7b16_m2f_head.pth'):
    !wget -q -O weights/dinov3_vit7b16_m2f_head.pth "{M2F_URL}"

print(f'Backbone: {os.path.getsize("weights/dinov3_vit7b16.pth")/1e9:.1f} GB')
print(f'M2F head: {os.path.getsize("weights/dinov3_vit7b16_m2f_head.pth")/1e9:.1f} GB')

## 2. Load the Full Segmentor

The Mask2Former segmentor architecture:

1. **DINOv3 ViT-7B backbone** (6.7B params, frozen) — extracts features from 4 evenly-spaced layers [10, 20, 30, 40]
2. **ViT-Adapter** (no injector) — reformats single-scale ViT features into a multi-scale FPN pyramid
3. **Mask2Former decoder** (927M params, pretrained) — predicts 100 mask queries + class labels

The final prediction combines masks and class probabilities:

```
output[c,h,w] = sum_q( class_prob[q,c] × mask_prob[q,h,w] )
```

In [ ]:
from src.mask2former_inference import load_mask2former_segmentor

segmentor = load_mask2former_segmentor(
    backbone_weights='weights/dinov3_vit7b16.pth',
    m2f_weights='weights/dinov3_vit7b16_m2f_head.pth',
    device='cuda',
    autocast_dtype=torch.bfloat16,  # Use bf16 to save memory
)

n_total = sum(p.numel() for p in segmentor.parameters())
n_trainable = sum(p.numel() for p in segmentor.parameters() if p.requires_grad)
print(f'Total parameters:     {n_total/1e9:.1f}B')
print(f'Trainable parameters: {n_trainable/1e6:.0f}M')
print(f'GPU memory used:      {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 3. Single Image Inference — Sanity Check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import v2
from src.mask2former_inference import slide_inference

# Load one validation image + ground truth
img_path = 'data/ADEChallengeData2016/images/validation/ADE_val_00000001.jpg'
ann_path = 'data/ADEChallengeData2016/annotations/validation/ADE_val_00000001.png'
img_pil = Image.open(img_path).convert('RGB')
gt = np.array(Image.open(ann_path))

# Preprocess: resize to 896 and normalize (matching paper inference settings)
transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((896, 896), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

img_tensor = transform(img_pil).unsqueeze(0).cuda()  # (1, 3, 896, 896)
print(f'Input shape: {img_tensor.shape}')

# Sliding window inference (crop=896, stride=596)
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
    logits = slide_inference(
        img_tensor, segmentor,
        num_classes=150,
        crop_size=(896, 896),
        stride=(596, 596),
    )

pred = logits.argmax(dim=1).squeeze().cpu().numpy()
print(f'Prediction shape: {pred.shape}')
print(f'Classes predicted: {len(np.unique(pred))}')

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_pil)
axes[0].set_title('Input Image')
axes[1].imshow(pred, cmap='Spectral', vmin=0, vmax=150)
axes[1].set_title('M2F Prediction')
axes[2].imshow(gt, cmap='Spectral', vmin=0, vmax=150)
axes[2].set_title('Ground Truth')
for ax in axes:
    ax.axis('off')
plt.suptitle('Mask2Former + DINOv3 ViT-7B — Single Image', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Full Validation Set Evaluation

Run sliding window inference on all 2,000 ADE20k validation images.
This takes approximately 2-4 hours on a single H100.

In [ ]:
from src.data_utils import ADE20kDataset, SegmentationEvalTransform
from src.evaluation import compute_iou_per_image, aggregate_metrics
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Validation dataset at resolution 896 (matching paper)
val_dataset = ADE20kDataset(
    root='data/ADEChallengeData2016',
    split='val',
    transform=SegmentationEvalTransform(img_size=896),
)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4)
print(f'Validation set: {len(val_dataset)} images at 896x896')

In [ ]:
# Full evaluation
NUM_CLASSES = 150
all_results = []

segmentor.eval()
for i, (images, targets) in enumerate(tqdm(val_loader, desc='M2F Inference')):
    images = images.cuda()

    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        logits = slide_inference(
            images, segmentor,
            num_classes=NUM_CLASSES,
            crop_size=(896, 896),
            stride=(596, 596),
        )

    # Resize prediction to match ground truth
    if logits.shape[-2:] != targets.shape[-2:]:
        logits = torch.nn.functional.interpolate(
            logits.cuda(), size=targets.shape[-2:],
            mode='bilinear', align_corners=False
        )

    pred = logits.argmax(dim=1)
    stats = compute_iou_per_image(pred, targets.cuda(), NUM_CLASSES)
    all_results.append(stats)

    if (i + 1) % 100 == 0:
        interim = aggregate_metrics(all_results)
        print(f'  [{i+1}/{len(val_loader)}] Running mIoU: {interim["mIoU"]:.2f}%')

# Final results
final = aggregate_metrics(all_results)
print(f'\n{"="*50}')
print(f'FINAL RESULTS (single-scale, no TTA)')
print(f'{"="*50}')
for k, v in final.items():
    print(f'  {k:>12s}: {v:.2f}%')
print(f'{"="*50}')
print(f'Paper (single-scale): 62.6 mIoU')
print(f'Paper (with TTA):     63.0 mIoU')

## 5. Qualitative Results Gallery

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])
indices = [0, 50, 100, 200, 500, 800, 1200, 1500]

fig, axes = plt.subplots(len(indices), 3, figsize=(15, 5 * len(indices)))

for row, idx in enumerate(indices):
    img_t, tgt_t = val_dataset[idx]
    img_t = img_t.unsqueeze(0).cuda()

    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        logits = slide_inference(img_t, segmentor, 150, (896, 896), (596, 596))
    pred = logits.argmax(1).squeeze().cpu().numpy()

    img_vis = np.clip(img_t[0].cpu().permute(1, 2, 0).float().numpy() * STD + MEAN, 0, 1)
    gt_vis = tgt_t.numpy()

    axes[row, 0].imshow(img_vis);    axes[row, 0].set_title('Image')
    axes[row, 1].imshow(pred, cmap='Spectral', vmin=0, vmax=150); axes[row, 1].set_title('Prediction')
    axes[row, 2].imshow(gt_vis, cmap='Spectral', vmin=0, vmax=150); axes[row, 2].set_title('Ground Truth')
    for ax in axes[row]:
        ax.axis('off')

plt.suptitle('Mask2Former + DINOv3 ViT-7B — ADE20k Validation Samples', fontsize=16)
plt.tight_layout()
os.makedirs('results/m2f_inference', exist_ok=True)
plt.savefig('results/m2f_inference/gallery.png', dpi=100)
plt.show()

## 6. Comparison: Linear Probe vs Mask2Former

| Aspect | Linear Probe (6.1.2) | Mask2Former (6.3.2) |
|--------|---------------------|---------------------|
| Decoder params | ~153K | 927M |
| Training needed | Yes (40K iters, ~2h) | No (pretrained) |
| Backbone | ViT-L or ViT-7B | ViT-7B only |
| Inference resolution | 512×512 | 896×896 |
| mIoU (ADE20k) | ~55 | ~63 |
| GPU requirement | T4 16GB (ViT-L) | H100 80GB |
| What it proves | Features are linearly separable | Full SOTA with frozen backbone |

Both approaches share the same core insight: **DINOv3's frozen features are so good**
that even a linear layer achieves impressive results, and a sophisticated decoder reaches
state-of-the-art — all without ever fine-tuning the backbone.